In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "00-foundations/transformers/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# The Original Transformer — Code Walkthrough

A minimal, runnable encoder-decoder Transformer (Vaswani et al., 2017), built one
component at a time and trained on a toy **"reverse the sequence"** task.

~200 lines of plain PyTorch. Every component has a *see-it-work* cell so the tensor
shapes are visible. Read top to bottom; run each cell.

**The two-line summary of the whole thing:**
- **Attention** mixes information *across positions* (tokens look at each other).
- **The feed-forward net** mixes information *across features* (per-token processing).

Everything else — masks, norms, positional encodings — is in service of those two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print("torch", torch.__version__)

## 1. Scaled dot-product attention

A **differentiable soft dictionary lookup**. A query is compared against every key
(dot product = similarity); softmax turns those similarities into weights that sum to
1; the output is the weighted average of the values.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The $\sqrt{d_k}$ keeps the scores from growing with dimension and saturating the
softmax. A `mask` of 0s blocks positions (set to $-\infty$ before softmax → 0 weight).

In [ ]:
def attention(Q, K, V, mask=None):
    # Q, K, V: (batch, heads, seq, d_k).  mask: 1 = keep, 0 = block.
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)      # (B, H, Tq, Tk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)                    # each row sums to 1
    return weights @ V, weights

In [ ]:
# see it work
B, H, T, d_k = 1, 1, 4, 8
Q, K, V = torch.randn(B, H, T, d_k), torch.randn(B, H, T, d_k), torch.randn(B, H, T, d_k)

out, w = attention(Q, K, V)
print("output shape :", tuple(out.shape))
print("weights shape:", tuple(w.shape))
print("row sums     :", w.sum(-1).flatten().tolist())   # all 1.0

# with a causal mask, token i cannot attend to j > i -> upper triangle is 0
cm = torch.tril(torch.ones(1, T, T)).unsqueeze(1)
_, wm = attention(Q, K, V, cm)
print("\ncausal weights (rows = queries, cols = keys):")
print(wm[0, 0].round(decimals=2))

## 2. Multi-head attention

One attention pattern can only express one kind of relationship. So split the model
dimension into `h` heads, run attention independently in each, concatenate, and mix
the results with a final projection `W_o`. Nearly free — you split the dimensions
rather than duplicating them.

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h, self.d_k = h, d_model // h
        self.W_q = nn.Linear(d_model, d_model)   # a "learned matrix" = W_Q etc.
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        # project, then reshape (B, T, d_model) -> (B, h, T, d_k)
        split = lambda x, W: W(x).view(B, -1, self.h, self.d_k).transpose(1, 2)
        Q, K, V = split(q, self.W_q), split(k, self.W_k), split(v, self.W_v)
        if mask is not None:
            mask = mask.unsqueeze(1)             # add head axis -> broadcast
        out, _ = attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.W_o(out)

In [ ]:
# see it work: shape goes in as d_model and comes out as d_model
mha = MultiHead(d_model=32, h=4)
x = torch.randn(2, 5, 32)          # (batch, seq, d_model)
print("in :", tuple(x.shape))
print("out:", tuple(mha(x, x, x).shape))   # self-attention: q = k = v = x

## 3. Position-wise feed-forward network

Attention is a weighted average — a linear op that moves information but builds no new
features. The FFN is the nonlinear per-token processing: expand, ReLU, project back.
Same weights at every position. (This holds ~2/3 of the model's parameters.)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 512 -> 2048 in the paper
            nn.ReLU(),
            nn.Linear(d_ff, d_model),   # back to 512
        )

    def forward(self, x):
        return self.net(x)

## 4. Positional encoding

Attention is **order-blind** — it sees a *set* of tokens. "dog bites man" and
"man bites dog" would be identical. Fix: add position information to the embeddings.
The paper uses fixed sines/cosines of geometrically increasing wavelength — fast waves
encode fine local position, slow waves encode coarse position.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)   # even dimensions: sine
        pe[:, 1::2] = torch.cos(pos * div)   # odd  dimensions: cosine
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]    # add, don't concatenate

In [ ]:
# see it work: each row is one position's encoding vector
pe = PositionalEncoding(d_model=64, max_len=100)
print("pe buffer shape:", tuple(pe.pe.shape))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3))
    plt.imshow(pe.pe[0, :40].T, aspect='auto', cmap='RdBu')
    plt.xlabel("position"); plt.ylabel("dimension"); plt.title("positional encoding")
    plt.colorbar(); plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## 5. The sublayer wrapper: residual + normalisation

Each sublayer is wrapped as `LayerNorm(x + Sublayer(x))`:
- **Residual `x + …`** — each sublayer computes an *adjustment*, giving gradients a
  clean path and keeping the vector shape fixed throughout.
- **LayerNorm** — rescales each token vector to a stable range.

> This is **post-norm**, as in the original paper. Essentially every model since ~2019
> uses **pre-norm** (`x + Sublayer(LayerNorm(x))`) instead — the first thing that changed.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.attn = MultiHead(d_model, h)
        self.ff = FeedForward(d_model, d_ff)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.n1(x + self.attn(x, x, x, mask))   # self-attention
        x = self.n2(x + self.ff(x))                 # feed-forward
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.self_attn = MultiHead(d_model, h)      # over output-so-far (masked)
        self.cross_attn = MultiHead(d_model, h)     # queries=decoder, keys/values=encoder
        self.ff = FeedForward(d_model, d_ff)
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d_model) for _ in range(3))

    def forward(self, x, enc, self_mask=None, cross_mask=None):
        x = self.n1(x + self.self_attn(x, x, x, self_mask))     # 1. masked self-attn
        x = self.n2(x + self.cross_attn(x, enc, enc, cross_mask))  # 2. cross-attn into encoder
        x = self.n3(x + self.ff(x))                             # 3. feed-forward
        return x

## 6. Causal mask

When generating, the model must not see the future. Set scores for positions `j > i`
to $-\infty$ so they get exactly 0 weight after softmax. The payoff: at training time
you process the whole target in **one parallel pass** and get a valid next-token
prediction at *every* position at once. (This is the reason Transformers replaced RNNs.)

In [ ]:
def causal_mask(T):
    return torch.tril(torch.ones(1, T, T)).long()   # (1, T, T): 1 = keep, 0 = future

print(causal_mask(5)[0])

## 7. The full model

Embed tokens → add positions → encoder stack → decoder stack (attending to the encoder)
→ linear projection to vocabulary logits.

> Two authentic details kept minimal: embeddings are scaled by $\sqrt{d_{model}}$, and
> real data needs a **padding mask** (skipped here — our sequences are fixed-length).

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, d_ff=128, layers=2, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.enc = nn.ModuleList([EncoderLayer(d_model, h, d_ff) for _ in range(layers)])
        self.dec = nn.ModuleList([DecoderLayer(d_model, h, d_ff) for _ in range(layers)])
        self.out = nn.Linear(d_model, vocab)

    def encode(self, src):
        x = self.pos(self.emb(src) * math.sqrt(self.d_model))
        for layer in self.enc:
            x = layer(x)
        return x

    def decode(self, tgt, enc):
        x = self.pos(self.emb(tgt) * math.sqrt(self.d_model))
        mask = causal_mask(tgt.size(1)).to(tgt.device)
        for layer in self.dec:
            x = layer(x, enc, mask, None)
        return x

    def forward(self, src, tgt):
        return self.out(self.decode(tgt, self.encode(src)))

## 8. Toy task — reverse a digit sequence

`[5, 6, 8, 0]` → `[0, 8, 6, 5]`. Tokens 0–9 are digits; token 10 is `<bos>` (the
decoder's start symbol). During training we use **teacher forcing**: the decoder input
is `<bos>` + the target shifted right, and it predicts the target.

In [ ]:
DIGITS, BOS, VOCAB, L = 10, 10, 11, 8

def make_batch(n):
    src = torch.randint(0, DIGITS, (n, L))
    tgt = torch.flip(src, dims=[1])                         # the reversed target
    dec_in = torch.cat([torch.full((n, 1), BOS), tgt[:, :-1]], dim=1)
    return src, dec_in, tgt

@torch.no_grad()
def greedy(model, src):
    # generate one token at a time (inference is sequential, unlike training)
    enc = model.encode(src)
    ys = torch.full((src.size(0), 1), BOS)
    for _ in range(L):
        logits = model.out(model.decode(ys, enc)[:, -1])   # last position only
        ys = torch.cat([ys, logits.argmax(-1, keepdim=True)], dim=1)
    return ys[:, 1:]                                        # drop <bos>

src, dec_in, tgt = make_batch(1)
print("source :", src[0].tolist())
print("dec_in :", dec_in[0].tolist(), "  (<bos>=10, then target shifted right)")
print("target :", tgt[0].tolist())

## 9. Train

One forward pass yields a prediction at every position, so one batch gives
`batch × seq` training signals. Watch loss fall and token-accuracy climb to ~1.0.

In [ ]:
model = Transformer(VOCAB)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

for step in range(1, 2001):
    src, dec_in, tgt = make_batch(64)
    logits = model(src, dec_in)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), tgt.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()

    if step % 250 == 0:
        s, _, t = make_batch(200)
        acc = (greedy(model, s) == t).float().mean().item()
        print(f"step {step:4d}   loss {loss.item():.3f}   token-acc {acc:.3f}")

In [ ]:
# see it reverse an unseen sequence
src, _, tgt = make_batch(1)
print("input   :", src[0].tolist())
print("model   :", greedy(model, src)[0].tolist())
print("expected:", tgt[0].tolist())

## Recap

1. **Attention** = differentiable soft lookup: compare a query to all keys, softmax the
   similarities, return a weighted average of the values.
2. A block does two things: **attention mixes across positions**, the **FFN mixes across
   features**. Everything stacks these.
3. **Residual + norm** make the stack deep enough to be useful.
4. **Positional encodings** are mandatory — attention itself is order-blind.
5. **Causal masking** extracts a training signal at every position in one parallel pass —
   the reason Transformers replaced RNNs.

Now open **`02_transformer_exercises.ipynb`** and rebuild the five core pieces from
scratch, with tests that tell you when each is right.